# Dub a video with IndicF5 — Kaggle

This notebook is **step 2 of 3**. It does one job: take the bundle your laptop
produced, run the voice model on a GPU, and hand you a results zip.

```
  1. your laptop     python -m src.cli ...            ->  tts_bundle.zip
  2. THIS NOTEBOOK   upload the zip, synthesize        ->  <job>_result.zip
  3. your laptop     python -m src.cli --from-stage import  ->  dubbed.mp4
```

Everything expensive is already done before you get here. Translation, timing
and segmentation ran on your laptop; this side only speaks the text it is
given, at the durations it is told.

**You need, before starting:**

| | |
|---|---|
| `tts_bundle.zip` | produced by step 1 on your laptop |
| a Hugging Face token | free, from huggingface.co/settings/tokens |
| access to `ai4bharat/IndicF5` | it is a gated repo — open the model page and accept the terms once |

Runtime is about **3–6 minutes on a T4** for a 60-second video, most of it
model download on the first run.

## 0. Turn on the GPU and the internet

Open the **⋮ menu at the top right → Accelerator → GPU T4 x2**, then
**Settings → Internet → On**.

Internet is off by default on Kaggle and the install below needs it.

Run the cell. **Expected:** a line naming a Tesla T4. If it prints nothing or
errors, the accelerator is still off.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 1. Add your bundle as a Dataset

**This is the Kaggle-specific step.** Kaggle has no direct file upload into a
running notebook — files come in as a *Dataset*.

1. In the right-hand sidebar, click **+ Add Input**
2. Click **Upload** → **New Dataset**
3. Drag in your `tts_bundle.zip`
4. Give it any title, e.g. `dubbing-bundles`, and click **Create**
5. Wait for it to finish, then make sure it appears under **Input** in the sidebar

You can upload several bundles into one dataset and this notebook will find
all of them.

> **If you change the zip later**, upload a *new version* of the dataset and
> re-attach it. Kaggle pins a dataset version to the notebook, so an edited
> file does not appear until you do.

Run the cell. **Expected:** your zip (or its extracted contents) listed under
`/kaggle/input/`.

In [ ]:
!ls -R /kaggle/input 2>/dev/null | head -30 || echo "no input attached yet — do step 1 above"

## 2. Your Hugging Face token

IndicF5 is a **gated** repo: you must be signed in, and you must have accepted
its terms at least once at
<https://huggingface.co/ai4bharat/IndicF5>.

**On Kaggle, tokens live in Add-ons → Secrets** (this is the main difference
from Colab, which uses its own Secrets panel and a different import):

1. **Add-ons → Secrets** in the top menu
2. **Add a new secret**
3. Label: `HF_TOKEN` — exactly that, it is case-sensitive
4. Value: your token from huggingface.co/settings/tokens (a `read` token is enough)
5. Tick the checkbox to attach it to this notebook

Run the cell. **Expected:** `logged in as <your-username>`.

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, whoami

login(token=UserSecretsClient().get_secret("HF_TOKEN"))
print("logged in as", whoami()["name"])

## 3. Clone and install

Two installs, in this order — IndicF5 first, then our pins, so ours are the
ones that survive.

> **You will see a red dependency-conflict line about `f5-tts` and `numpy`.
> That is expected and is not a failure.** f5-tts declares a numpy ceiling
> inherited from an older scientific stack; librosa's numba needs a newer one.
> We deliberately override it, and IndicF5 runs correctly on numpy 2.2.
> `requirements-gpu.txt` explains it in full.

Takes about 2 minutes. **Expected:** the two `tail -2` lines, then the restart
instruction.

In [ ]:
import os

os.chdir("/kaggle/working")
!rm -rf indic-dub-pipeline
!git clone -q -b test https://github.com/ayushk1233/indic-dub-pipeline.git
os.chdir("/kaggle/working/indic-dub-pipeline")
!git log --oneline -1

!pip install -q git+https://github.com/ai4bharat/IndicF5.git 2>&1 | tail -2
!pip install -q -r requirements-gpu.txt 2>&1 | tail -2

print("\n" + "=" * 60)
print("INSTALLED. Now restart: Run -> Restart session.")
print("Then continue at section 4. Do NOT re-run this cell.")
print("=" * 60)

---

# ⛔ RESTART THE SESSION NOW — not optional

**Run → Restart session**, then carry on at section 4 below.

The install changed `numpy` and `transformers`. Anything Python already
imported in this session is still holding the *old* module objects, and the
failure does not show up here — it surfaces much later as a confusing
`ImportError` from inside the model loader, or as a model class that has
silently vanished.

**Do not re-run section 3 after restarting.** The clone and the packages are
already on disk. Everything from section 4 onward re-establishes what it needs.

---

## 4. Verify the environment (after the restart)

Run the cell. **Expected:** every line reports a version and the last line says
`environment OK`. If `numpy` is below 2.1 or `transformers` is 5.x, the restart
did not happen — restart and re-run this cell only.

In [ ]:
import os, sys

os.chdir("/kaggle/working/indic-dub-pipeline")
sys.path.insert(0, "/kaggle/working/indic-dub-pipeline")

import numpy, torch, transformers

print(f"numpy         {numpy.__version__:<12} (needs >=2.1,<2.3)")
print(f"torch         {torch.__version__:<12} cuda={torch.cuda.is_available()}")
print(f"transformers  {transformers.__version__:<12} (needs >=4.57,<5)")
print(f"gpu           {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")

assert torch.cuda.is_available(), "no GPU — check the Accelerator setting, then restart"
assert numpy.__version__ >= "2.1", "stale numpy — you did not restart the session"

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(token=UserSecretsClient().get_secret("HF_TOKEN"))

print("\nenvironment OK")

## 5. Find the bundle

Handles either shape Kaggle gives you — a `.zip` left packed, or a directory it
auto-extracted — and copies each bundle somewhere writable, because
`/kaggle/input` is read-only and the worker writes its output inside the bundle.

Run the cell. **Expected:** one line per bundle, naming its segment count,
language and job id. If it finds nothing, go back to section 1.

In [ ]:
import glob, json, shutil, zipfile
from pathlib import Path

WORK = Path("/kaggle/working")
BUNDLES = {}

def register(src):
    request = src / "request" / "synthesis_request.json"
    if not request.exists():
        return
    data = json.loads(request.read_text(encoding="utf-8"))
    job = data["job_id"]
    dest = WORK / f"{job}_bundle"
    shutil.rmtree(dest, ignore_errors=True)
    shutil.copytree(src, dest)
    BUNDLES[job] = dest

for archive in glob.glob("/kaggle/input/**/*.zip", recursive=True):
    tmp = WORK / "_unzipped" / Path(archive).stem
    shutil.rmtree(tmp, ignore_errors=True)
    with zipfile.ZipFile(archive) as z:
        z.extractall(tmp)
    for manifest in tmp.rglob("manifest.json"):
        register(manifest.parent)

for manifest in glob.glob("/kaggle/input/**/manifest.json", recursive=True):
    register(Path(manifest).parent)

if not BUNDLES:
    raise SystemExit("No bundle found under /kaggle/input — see section 1.")

for job, path in sorted(BUNDLES.items()):
    data = json.loads((path / "request" / "synthesis_request.json").read_text(encoding="utf-8"))
    print(f"{job:<12} {len(data['segments']):>3} segments   language={data['language']}   -> {path}")

## 6. Synthesize

This is the part that needs the GPU. The worker loads IndicF5 once, then walks
the bundle segment by segment.

**Expected output**, per segment: the text it was asked for, the duration it was
told to hit, and `done`. It writes its result file after *every* segment, so if
the session dies halfway the finished clips survive and re-running picks up a
complete file.

First run downloads roughly 2 GB of weights. Later runs reuse the cache.

**If a segment says `failed`**, the run continues — one bad segment does not
lose the job. The error is recorded in `logs/errors.log` inside the bundle.

In [ ]:
for job, path in sorted(BUNDLES.items()):
    print(f"\n{'=' * 24} {job} {'=' * 24}")
    !python -m colab.indicf5_worker --bundle {path}

## 7. Download the results

Run the cell, then click each link. **Expected:** one `<job>_result.zip` per
bundle, a few MB each.

Save them somewhere you can find — step 3 on your laptop needs them.

In [ ]:
import shutil
from IPython.display import FileLink, display

for job, path in sorted(BUNDLES.items()):
    archive = shutil.make_archive(str(WORK / f"{job}_result"), "zip", path / "output")
    size = Path(archive).stat().st_size / 1e6
    print(f"{job:<12} {size:>6.1f} MB")
    display(FileLink(f"{job}_result.zip"))

## 8. Listen before you leave (optional, recommended)

Worth 30 seconds. Every timing metric can read perfectly green while the audio
is wrong, so the only check that catches a bad run early is your ear.

Play a few clips. They should say the text printed above them, in the
speaker's voice, with nothing invented at the start.

In [ ]:
import json
from IPython.display import Audio, display

job = sorted(BUNDLES)[0]        # change this to hear another bundle
path = BUNDLES[job]

asked = {s["segment_id"]: s["text"] for s in json.loads(
    (path / "request" / "synthesis_request.json").read_text(encoding="utf-8"))["segments"]}
result = json.loads((path / "output" / "synthesis_result.json").read_text(encoding="utf-8"))

for segment in result["segments"][:5]:
    index = segment["segment_id"]
    print(f"[{index:>2}] {segment['status']}  {segment.get('duration', 0):.2f}s")
    print(f"     {asked[index]}")
    if segment["status"] == "done":
        display(Audio(str(path / segment["audio_path"])))

---

## Back to your laptop

Unzip each result into its job's bundle directory, then run the third command:

```bash
unzip -o ~/Downloads/demo_result.zip -d artifacts/demo/tts_bundle/output/

python -m src.cli \
  --input myvideo.mp4 \
  --job-id demo \
  --target-lang hi \
  --from-stage import
```

That assembles the clips onto the original timeline and writes
`artifacts/demo/dubbed.mp4`.

---

## Troubleshooting

**`OSError: You are trying to access a gated repo`**
You have a token but have not accepted the model terms. Open
<https://huggingface.co/ai4bharat/IndicF5> while signed in, accept, re-run
section 4.

**`KeyError: 'HF_TOKEN'` or `Secret not found`**
The secret is not attached to *this* notebook. Add-ons → Secrets → tick the
box next to `HF_TOKEN`. Attaching is separate from creating.

**`ImportError` mentioning `transformers` or a missing model class**
You did not restart after section 3, or you re-ran section 3 after restarting.
Run → Restart session, then start again at section 4.

**A red `pip` line about `f5-tts` requiring `numpy<=1.26.4`**
Expected. Not a failure. See the note in section 3.

**`No bundle found under /kaggle/input`**
The dataset is not attached, or you attached it after this notebook started.
Check the Input list in the sidebar; re-run section 5.

**`CUDA out of memory`**
Another notebook is holding the GPU. Run → Restart session and re-run from
section 4 — the clone and packages are still on disk.

**Kaggle GPU quota exhausted**
30 hours per week, reset weekly. The Colab notebook in this same folder does
the identical job if you need to finish now.